In [11]:
import joblib
import pandas as pd
import shap

import numpy as np

import sys
import os

sys.path.append(os.path.abspath("..\\feature_engineering"))

from feature_engineering import feature_engineering as fe

fe_path = os.path.abspath("..\\feature_engineering")
features = joblib.load(os.path.join(fe_path, "features.pkl"))
print("✅ Feature list loaded:", features)

model = joblib.load(os.path.join("..\\prediction", "multi_output_xgb_model.pkl"))

# -------------------------------
# LOAD TRAIN DATA (FOR SHAP)
# -------------------------------
train_unscaled = pd.read_csv(os.path.join(fe_path, "train_unscaled_processed.csv"))

target_cols = ["Fatigue","FutureHealthRisk","DiabetesRisk","AnemiaRisk","PCOSRisk"]

X_train = train_unscaled.drop(columns=target_cols)



✅ Feature list loaded: ['ScreenTime', 'SleepHours', 'LateNightUsage', 'ActivityLevel', 'DietQuality', 'SittingTime', 'InactivityPeriods', 'StressLevel', 'Gender', 'MealsPerDay', 'CalorieIntake', 'BMI', 'dopamine_score', 'lifestyle_risk', 'health_score', 'bmi_category']


example

In [17]:
explainers = []

for i in range(len(target_cols)):
    explainer = shap.TreeExplainer(model.estimators_[i])
    explainers.append(explainer)

print("✅ SHAP Explainers Ready")

✅ SHAP Explainers Ready


In [18]:
def predict_and_explain(user_input):

    # Convert input to DataFrame
    df = pd.DataFrame([user_input])

    # Apply feature engineering
    df = fe(df)

    # 🔥 CRITICAL: Align features with model
    expected_features = model.estimators_[0].feature_names_in_

    # Add missing columns if any
    for col in expected_features:
        if col not in df.columns:
            df[col] = 0

    # Ensure correct order
    df = df[expected_features]

    # -------------------------------
    # Prediction
    # -------------------------------
    preds = model.predict(df)[0]

    results = {}

    # -------------------------------
    # SHAP Explanation
    # -------------------------------
    for i, target in enumerate(target_cols):

        explainer = explainers[i]

        shap_values = explainer(df, check_additivity=False)

        # Get predicted class
        pred_class = int(preds[i])

        # Extract correct SHAP values (multi-class fix)
        vals = shap_values.values[0, :, pred_class]

        # Get top contributing features
        feature_names = df.columns
        idx = np.argsort(np.abs(vals))[::-1][:3]

        top_features = [(feature_names[j], float(vals[j])) for j in idx]

        results[target] = {
            "prediction": pred_class,
            "top_features": top_features
        }

    return results


🔍 Prediction + Explanation:

Fatigue:
Prediction: 2
Top Factors: [('StressLevel', 2.8118982315063477), ('ScreenTime', 2.2092974185943604), ('SleepHours', 2.1616811752319336)]

FutureHealthRisk:
Prediction: 2
Top Factors: [('ActivityLevel', -3.919912099838257), ('health_score', 2.6882920265197754), ('StressLevel', 0.9695122241973877)]

DiabetesRisk:
Prediction: 1
Top Factors: [('ActivityLevel', 0.9570642709732056), ('CalorieIntake', 0.74544757604599), (np.str_('SedentaryIndex'), 0.4730149805545807)]

AnemiaRisk:
Prediction: 1
Top Factors: [('DietQuality', 1.1616644859313965), ('health_score', -0.6518716216087341), ('CalorieIntake', 0.45569151639938354)]

PCOSRisk:
Prediction: 1
Top Factors: [('health_score', -1.2085374593734741), ('ActivityLevel', 1.0448706150054932), (np.str_('SedentaryIndex'), -0.27033889293670654)]


In [22]:
def generate_human_insights(results):

    label_map = {0: "Low", 1: "Medium", 2: "High"}

    insights = {}

    for target, data in results.items():

        pred_label = label_map[data["prediction"]]

        reasons = []
        suggestions = []

        for feature, value in data["top_features"]:

            # -------------------------------
            # REASON GENERATION
            # -------------------------------
            if value > 0:
                reasons.append(f"{feature} is increasing the risk")
            else:
                reasons.append(f"{feature} is helping reduce the risk")

            # -------------------------------
            # RECOMMENDATIONS (RULE-BASED)
            # -------------------------------
            if feature == "SleepHours" or feature == "SleepScore":
                suggestions.append("Improve sleep to 7–8 hours")

            elif feature == "StressLevel" or feature == "StressScore":
                suggestions.append("Practice stress management (meditation, exercise)")

            elif feature == "ScreenTime" or feature == "DigitalLoad":
                suggestions.append("Reduce screen time, especially at night")

            elif feature == "ActivityLevel" or feature == "ActivityScore":
                suggestions.append("Increase daily physical activity")

            elif feature == "DietQuality":
                suggestions.append("Improve diet quality (balanced nutrition)")

            elif feature == "SedentaryIndex":
                suggestions.append("Reduce long sitting periods")

            elif feature == "CalorieIntake":
                suggestions.append("Maintain a balanced calorie intake")

        # remove duplicates
        suggestions = list(set(suggestions))

        insights[target] = {
            "Prediction": pred_label,
            "Reasons": reasons,
            "Recommendations": suggestions
        }

    return insights

In [24]:
sample_user = {
    "ScreenTime": 6,
    "SleepHours": 5,
    "LateNightUsage": 1,
    "ActivityLevel": 2,
    "DietQuality": 1,
    "SittingTime": 8,
    "InactivityPeriods": 3,
    "StressLevel": 8,
    "Gender": "Male",
    "MealsPerDay": 2,
    "CalorieIntake": 2500,
    "BMI": 28,
    "dopamine_score": 7
}

result = predict_and_explain(sample_user)
insights = generate_human_insights(result)
print("\n🧠 AI Health Insights:\n")

for target, data in insights.items():

    print(f"\n🔹 {target}")
    print("Prediction:", data["Prediction"])

    print("\nWhy?")
    for r in data["Reasons"]:
        print("-", r)

    print("\nWhat to do?")
    for s in data["Recommendations"]:
        print("-", s)


🧠 AI Health Insights:


🔹 Fatigue
Prediction: High

Why?
- StressLevel is increasing the risk
- ScreenTime is increasing the risk
- SleepHours is increasing the risk

What to do?
- Reduce screen time, especially at night
- Practice stress management (meditation, exercise)
- Improve sleep to 7–8 hours

🔹 FutureHealthRisk
Prediction: High

Why?
- ActivityLevel is helping reduce the risk
- health_score is increasing the risk
- StressLevel is increasing the risk

What to do?
- Increase daily physical activity
- Practice stress management (meditation, exercise)

🔹 DiabetesRisk
Prediction: Medium

Why?
- ActivityLevel is increasing the risk
- CalorieIntake is increasing the risk
- SedentaryIndex is increasing the risk

What to do?
- Reduce long sitting periods
- Increase daily physical activity
- Maintain a balanced calorie intake

🔹 AnemiaRisk
Prediction: Medium

Why?
- DietQuality is increasing the risk
- health_score is helping reduce the risk
- CalorieIntake is increasing the risk

What